In [ ]:
!pip install datasets torch tqdm

In [ ]:
from datasets import load_dataset

# Load dataset
ds = load_dataset("jonathanyly/python-code-dataset-500k")

# Take small subset (5000 samples)
train_data = ds["train"].select(range(5000))

# Print the content of the 'output' column for the first sample
print(train_data[0]["output"])


```python
for i in range(10):  # First digit
    for j in range(10):  # Second digit
        for k in range(10):  # Third digit
            # Checking for the conditions
            if i != 5 and j != 5 and k != 5 and i != j and i != k and j != k:
                print(i, j, k)
```


In [ ]:
import tokenize
import io
from tokenize import TokenError

def tokenize_with_indent(code):
    lines = code.strip().splitlines()

    # ----------------------------
    # Handle Markdown Code Blocks
    # ----------------------------
    if len(lines) > 0 and lines[0].strip().startswith("```"):
        first_line_content = lines[0].strip().lstrip("```").strip().lower()

        # Ignore non-Python markdown blocks
        if first_line_content not in ["", "python"]:
            return []

        # Strip markdown delimiters
        if lines[-1].strip() == "```":
            code = "\n".join(lines[1:-1])

    # ----------------------------
    # Skip Raw HTML
    # ----------------------------
    if code.strip().lower().startswith("<!doctype html>"):
        return []

    tokens = []

    try:
        g = tokenize.generate_tokens(io.StringIO(code).readline)

        for toknum, tokval, _, _, _ in g:
            if toknum == tokenize.INDENT:
                tokens.append("<INDENT>")
            elif toknum == tokenize.DEDENT:
                tokens.append("<DEDENT>")
            elif toknum == tokenize.NEWLINE:
                tokens.append("<NEWLINE>")
            elif tokval.strip() != "":
                tokens.append(tokval)

    except TokenError:
        # 🔥 Handle incomplete code safely (VERY IMPORTANT)
        # Fallback to simple split for partial code
        tokens = code.replace("(", " ( ") \
                     .replace(")", " ) ") \
                     .replace(":", " : ") \
                     .split()

    except Exception:
        # Final fallback
        tokens = code.split()

    return tokens

In [ ]:
tokenized_data = []

for i, item in enumerate(train_data):
    try:
        tokens = tokenize_with_indent(item["output"])
        tokenized_data.append(tokens)
    except tokenize.TokenError as e:
        print(f"TokenError encountered for item {i}: {e}")
        print(f"Problematic code:\n{item['output']}")
        # Skip this item to continue processing others if possible
        continue
    except Exception as e:
        print(f"An unexpected error occurred for item {i}: {e}")
        print(f"Problematic code:\n{item['output']}")
        continue

# Ensure tokenized_data is not empty before trying to print
if tokenized_data:
    print(tokenized_data[0][:30])
else:
    print("No data was tokenized successfully.")

['for', 'i', 'in', 'range', '(', '10', ')', ':', '# First digit', '<NEWLINE>', '<INDENT>', 'for', 'j', 'in', 'range', '(', '10', ')', ':', '# Second digit', '<NEWLINE>', '<INDENT>', 'for', 'k', 'in', 'range', '(', '10', ')', ':']


In [ ]:
from collections import Counter

def build_vocab(tokenized_data, min_freq=2):
    counter = Counter()
    for tokens in tokenized_data:
        counter.update(tokens)

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab

vocab = build_vocab(tokenized_data)
vocab_size = len(vocab)

print("Vocab size:", vocab_size)

Vocab size: 19417


In [ ]:
import torch

def encode(tokens, vocab):
    return [vocab.get(t, vocab["<UNK>"]) for t in tokens]

encoded_data = [encode(tokens, vocab) for tokens in tokenized_data]

sequence_length = 20
inputs = []
targets = []

for seq in encoded_data:
    for i in range(len(seq) - sequence_length):
        inputs.append(seq[i:i+sequence_length])
        targets.append(seq[i+sequence_length])

X = torch.tensor(inputs)
y = torch.tensor(targets)

print(X.shape, y.shape)

torch.Size([956650, 20]) torch.Size([956650])


In [ ]:
import torch.nn as nn

class CodeLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        out = self.dropout(out[:, -1, :])
        logits = self.fc(out)
        return logits

model = CodeLSTM(vocab_size)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# ---------------------------
# Move model to device
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

# ---------------------------
# Custom Dataset (No sliding window)
# ---------------------------
class CodeDataset(Dataset):
    def __init__(self, encoded_sequences, seq_len=30):
        self.data = encoded_sequences
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]

        if len(seq) < self.seq_len + 1:
            seq = seq + [0] * (self.seq_len + 1 - len(seq))

        input_seq = torch.tensor(seq[:self.seq_len])
        target = torch.tensor(seq[self.seq_len])   # ✅ Only next token

        return input_seq, target
# ---------------------------
# Create Dataset
# ---------------------------
seq_len = 30
dataset = CodeDataset(encoded_data, seq_len)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# ---------------------------
# Loss & Optimizer
# ---------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 1

# ---------------------------
# Faster Training Loop
# ---------------------------
for epoch in range(epochs):
    total_loss = 0

    for batch_x, batch_y in tqdm(loader):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)

        # Reshape for sequence prediction
        outputs = outputs.view(-1, outputs.size(-1))
        batch_y = batch_y.view(-1)

        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Using device: cpu


100%|██████████| 157/157 [00:22<00:00,  7.05it/s]

Epoch 1, Loss: 5.5001


In [ ]:
import torch.nn as nn

class CodeLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        # Apply dropout and FC layer to each time step's output
        out = self.dropout(out) # out shape: (batch_size, seq_len, hidden_dim)
        logits = self.fc(out)  # logits shape: (batch_size, seq_len, vocab_size)
        return logits

model = CodeLSTM(vocab_size)

In [ ]:
import torch.nn.functional as F

def generate_top_k(model, seed_tokens, vocab, k=3):
    model.eval()

    encoded = torch.tensor([encode(seed_tokens, vocab)])
    logits = model(encoded)
    probs = F.softmax(logits, dim=-1)

    topk_probs, topk_indices = torch.topk(probs, k)

    suggestions = []
    inv_vocab = {v: k for k, v in vocab.items()}

    for i in range(k):
        token_id = topk_indices[0][i].item()
        suggestions.append({
            "suggestion": inv_vocab[token_id],
            "probability": round(topk_probs[0][i].item(), 4)
        })

    return suggestions

In [ ]:
import torch.nn.functional as F

def generate_top_k_fixed(model, seed_tokens, vocab, k=3):
    model.eval()

    encoded = torch.tensor([encode(seed_tokens, vocab)])

    # Get the model's output for the entire sequence
    all_logits = model(encoded)

    # We are interested in the prediction for the next token, based on the last token in the seed_tokens
    # So we take the logits corresponding to the last position in the sequence
    last_token_logits = all_logits[:, -1, :]

    probs = F.softmax(last_token_logits, dim=-1)

    topk_probs, topk_indices = torch.topk(probs, k)

    suggestions = []
    inv_vocab = {v: k for k, v in vocab.items()}

    for i in range(k):
        token_id = topk_indices[0][i].item()
        suggestions.append({
            "suggestion": inv_vocab[token_id],
            "probability": round(topk_probs[0][i].item(), 4)
        })

    return suggestions

seed = tokenized_data[0][:20]
print(generate_top_k_fixed(model, seed, vocab, k=3))

[{'suggestion': 'Caching', 'probability': 0.0001}, {'suggestion': 'retry_attempt', 'probability': 0.0001}, {'suggestion': '# [[1, 4, 7], [2, 5, 8], [3, 6, 9]]', 'probability': 0.0001}]


In [ ]:
import torch.nn.functional as F

def generate_full_word_suggestions(model, seed_tokens, vocab, k=3, num_tokens_to_generate=3):
    model.eval()
    inv_vocab = {v: k for k, v in vocab.items()}

    # Start with the initial top-k suggestions
    initial_suggestions = generate_top_k_fixed(model, seed_tokens, vocab, k=k)

    extended_suggestions = []

    for initial_pred in initial_suggestions:
        current_sequence = seed_tokens + [initial_pred['suggestion']]
        current_text = initial_pred['suggestion']
        current_prob = initial_pred['probability']

        # Generate subsequent tokens
        for _ in range(num_tokens_to_generate - 1):
            if not current_sequence:
                break
            # Predict only the single best next token based on the current sequence
            next_tokens = generate_top_k_fixed(model, current_sequence[-20:], vocab, k=1)
            if next_tokens:
                next_token_str = next_tokens[0]['suggestion']
                next_token_prob = next_tokens[0]['probability']

                # If the next token is a structural token or empty, consider it a breaking point
                if next_token_str in ['<NEWLINE>', '<INDENT>', '<DEDENT>', ' ', '(', ')', '[', ']', '{', '}', '.', ',', ':', ';', '='] or not next_token_str.strip():
                    break

                # Append the token and update probability (simplified product)
                current_sequence.append(next_token_str)
                current_text += next_token_str
                current_prob *= next_token_prob # Simplified probability combination
            else:
                break

        extended_suggestions.append({
            "suggestion": current_text,
            "probability": round(current_prob, 4)
        })

    # Sort by probability
    extended_suggestions = sorted(extended_suggestions, key=lambda x: x['probability'], reverse=True)

    return extended_suggestions


In [ ]:
user_code = input("Enter partial Python code: ")

tokens = tokenize_with_indent(user_code)
seed_tokens = tokens[-20:]

print(generate_full_word_suggestions(model, seed_tokens, vocab, k=3))

Enter partial Python code: for i
[{'suggestion': 'threading,weighted_medianHandling', 'probability': 0.0}, {'suggestion': '"0. Exit"customers_with_addressat', 'probability': 0.0}, {'suggestion': 'optimizationoversamplerbinary_list_to_integer', 'probability': 0.0}]


In [ ]:
# ==============================
# SINGLE CELL - CODE AUTO COMPLETION USING LSTM
# ==============================

import os
import io
import tokenize
import keyword
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader

# ------------------------------
# 1️⃣  CONFIG
# ------------------------------
SEQ_LENGTH = 5
EMBED_SIZE = 128
HIDDEN_SIZE = 256
BATCH_SIZE = 64
EPOCHS = 3
MIN_FREQ = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------
# 2️⃣  TOKENIZER
# ------------------------------
def tokenize_python_code(code):
    tokens = []
    reader = io.StringIO(code).readline
    for toknum, tokval, _, _, _ in tokenize.generate_tokens(reader):
        if tokval.strip() == "":
            continue
        if tokval in keyword.kwlist:
            tokens.append(tokval)
        elif tokval.isidentifier():
            tokens.append("<IDENTIFIER>")
        elif tokval.isnumeric():
            tokens.append("<NUMBER>")
        elif len(tokval) < 30:
            tokens.append(tokval)
    return tokens

# ------------------------------
# 3️⃣  LOAD DATA (Change path)
# ------------------------------
def load_python_files(folder_path):
    all_tokens = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".py"):
                try:
                    with open(os.path.join(root, file), "r", encoding="utf-8") as f:
                        code = f.read()
                        tokens = tokenize_python_code(code)
                        all_tokens.extend(tokens)
                except:
                    pass
    return all_tokens

# 👉 CHANGE THIS PATH TO YOUR PYTHON DATASET FOLDER
DATA_PATH = "python_dataset"

tokens = load_python_files(DATA_PATH)
print("Total tokens:", len(tokens))

# ------------------------------
# 4️⃣  BUILD VOCAB
# ------------------------------
counter = Counter(tokens)
vocab = {word for word, freq in counter.items() if freq >= MIN_FREQ}

word2idx = {word: idx+2 for idx, word in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {idx: word for word, idx in word2idx.items()}

print("Vocab size:", len(word2idx))

# ------------------------------
# 5️⃣  CREATE SEQUENCES
# ------------------------------
encoded = [word2idx.get(word, 1) for word in tokens]

inputs = []
targets = []

for i in range(len(encoded) - SEQ_LENGTH):
    inputs.append(encoded[i:i+SEQ_LENGTH])
    targets.append(encoded[i+SEQ_LENGTH])

# ------------------------------
# 6️⃣  DATASET
# ------------------------------
class CodeDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = torch.tensor(inputs)
        self.targets = torch.tensor(targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

dataset = CodeDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ------------------------------
# 7️⃣  LSTM MODEL
# ------------------------------
class CodeLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(CodeLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = CodeLSTM(len(word2idx), EMBED_SIZE, HIDDEN_SIZE).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ------------------------------
# 8️⃣  TRAINING
# ------------------------------
print("Training started...\n")

for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(dataloader):.4f}")

print("\nTraining Complete!")

# ------------------------------
# 9️⃣  PREDICTION FUNCTION
# ------------------------------
def predict_next_tokens(model, text, top_k=3):
    model.eval()
    tokens = tokenize_python_code(text)
    tokens = tokens[-SEQ_LENGTH:]

    encoded = [word2idx.get(word, 1) for word in tokens]
    if len(encoded) < SEQ_LENGTH:
        encoded = [0] * (SEQ_LENGTH - len(encoded)) + encoded

    input_tensor = torch.tensor([encoded]).to(DEVICE)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)
        top_probs, top_indices = torch.topk(probs, top_k)

    suggestions = []
    for prob, idx in zip(top_probs[0], top_indices[0]):
        suggestions.append({
            "suggestion": idx2word.get(idx.item(), "<UNK>"),
            "probability": round(prob.item(), 4)
        })

    return suggestions

# ------------------------------
# 🔟  MANUAL USER INPUT
# ------------------------------
while True:
    user_input = input("\nEnter partial Python code (or 'exit'): ")
    if user_input.lower() == "exit":
        break

    result = predict_next_tokens(model, user_input)
    print(result)

Total tokens: 0
Vocab size: 2


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [1]:
# =====================================================
# CODE AUTO-COMPLETION USING HUGGINGFACE DATASET
# WITH TOP-K ACCURACY EVALUATION
# =====================================================

!pip -q install datasets

import io
import tokenize
import keyword
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

# -----------------------
# CONFIGURATION
# -----------------------
SEQ_LENGTH = 6
EMBED_SIZE = 128
HIDDEN_SIZE = 256
BATCH_SIZE = 128
EPOCHS = 2
MIN_FREQ = 10   # reduce vocab noise
MAX_SAMPLES = 20000  # limit for faster Colab training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

# -----------------------
# LOAD DATASET
# -----------------------
dataset = load_dataset("code_search_net", "python", split=f"train[:{MAX_SAMPLES}]")
print("Dataset Loaded")
print("Columns:", dataset.column_names)

# -----------------------
# TOKENIZER
# -----------------------
def tokenize_python_code(code):
    tokens = []
    reader = io.StringIO(code).readline
    try:
        for toknum, tokval, _, _, _ in tokenize.generate_tokens(reader):
            if tokval.strip() == "":
                continue
            if tokval in keyword.kwlist:
                tokens.append(tokval)
            elif tokval.isidentifier():
                tokens.append("<IDENTIFIER>")
            elif tokval.isnumeric():
                tokens.append("<NUMBER>")
            elif len(tokval) < 30:
                tokens.append(tokval)
    except:
        pass
    return tokens

# -----------------------
# COLLECT TOKENS
# -----------------------
all_tokens = []

for item in dataset:
    code = item["func_code_string"]   # Correct column
    tokens = tokenize_python_code(code)
    all_tokens.extend(tokens)

print("Total tokens:", len(all_tokens))

# -----------------------
# BUILD VOCAB
# -----------------------
counter = Counter(all_tokens)
vocab = {w for w, f in counter.items() if f >= MIN_FREQ}

word2idx = {w: i+2 for i, w in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {i: w for w, i in word2idx.items()}

print("Vocab size:", len(word2idx))

# -----------------------
# CREATE SEQUENCES
# -----------------------
encoded = [word2idx.get(w, 1) for w in all_tokens]

inputs, targets = [], []
for i in range(len(encoded) - SEQ_LENGTH):
    inputs.append(encoded[i:i+SEQ_LENGTH])
    targets.append(encoded[i+SEQ_LENGTH])

print("Total sequences:", len(inputs))

# -----------------------
# TRAIN / VALID SPLIT
# -----------------------
split_index = int(0.9 * len(inputs))

train_x = inputs[:split_index]
train_y = targets[:split_index]
val_x = inputs[split_index:]
val_y = targets[split_index:]

# -----------------------
# DATASET CLASS
# -----------------------
class CodeDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_loader = DataLoader(CodeDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CodeDataset(val_x, val_y), batch_size=BATCH_SIZE)

# -----------------------
# MODEL
# -----------------------
class CodeLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_SIZE)
        self.lstm = nn.LSTM(EMBED_SIZE, HIDDEN_SIZE, batch_first=True)
        self.fc = nn.Linear(HIDDEN_SIZE, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = CodeLSTM(len(word2idx)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------
# TRAINING
# -----------------------
print("\nTraining...\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("Training Complete")

# -----------------------
# TOP-K ACCURACY FUNCTION
# -----------------------
def top_k_accuracy(model, loader, k):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            _, topk = torch.topk(outputs, k, dim=1)

            for i in range(len(y)):
                if y[i] in topk[i]:
                    correct += 1
                total += 1

    return correct / total

print("\nValidation Accuracy:")
print("Top-1:", round(top_k_accuracy(model, val_loader, 1), 4))
print("Top-3:", round(top_k_accuracy(model, val_loader, 3), 4))
print("Top-5:", round(top_k_accuracy(model, val_loader, 5), 4))

# -----------------------
# PREDEFINED USER TEST INPUTS
# -----------------------
test_samples = [
    "for i in",
    "if x >",
    "def add(",
    "while True",
]

def predict_next(text, k=3):
    model.eval()
    tokens = tokenize_python_code(text)
    tokens = tokens[-SEQ_LENGTH:]
    encoded = [word2idx.get(t, 1) for t in tokens]

    if len(encoded) < SEQ_LENGTH:
        encoded = [0]*(SEQ_LENGTH-len(encoded)) + encoded

    x = torch.tensor([encoded]).to(DEVICE)

    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        top_probs, top_idx = torch.topk(probs, k)

    results = []
    for p, i in zip(top_probs[0], top_idx[0]):
        results.append((idx2word[i.item()], round(p.item(),4)))
    return results

print("\nSample Predictions:\n")
for sample in test_samples:
    print("Input:", sample)
    print("Top-3:", predict_next(sample, 3))
    print()


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'torch'

In [2]:
# ==========================================
# INTERACTIVE USER TESTING
# ==========================================

def predict_next(text, k=5):
    model.eval()

    tokens = tokenize_python_code(text)
    tokens = tokens[-SEQ_LENGTH:]
    encoded = [word2idx.get(t, 1) for t in tokens]

    if len(encoded) < SEQ_LENGTH:
        encoded = [0]*(SEQ_LENGTH-len(encoded)) + encoded

    x = torch.tensor([encoded]).to(DEVICE)

    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        top_probs, top_idx = torch.topk(probs, k)

    results = []
    for p, i in zip(top_probs[0], top_idx[0]):
        results.append({
            "token": idx2word.get(i.item(), "<UNK>"),
            "probability": round(p.item(), 4)
        })
    return results


# Interactive loop
while True:
    user_input = input("\nEnter partial Python code (or type 'exit'): ")

    if user_input.lower() == "exit":
        break

    predictions = predict_next(user_input, k=5)
    print("Top-5 Suggestions:")
    for p in predictions:
        print(p)


Enter partial Python code (or type 'exit'): for i in 
Top-5 Suggestions:
{'token': '<IDENTIFIER>', 'probability': 0.8615}
{'token': '(', 'probability': 0.0512}
{'token': '<NUMBER>', 'probability': 0.0506}
{'token': '[', 'probability': 0.0265}
{'token': '{', 'probability': 0.0048}

Enter partial Python code (or type 'exit'): int num 
Top-5 Suggestions:
{'token': '[', 'probability': 0.3599}
{'token': '(', 'probability': 0.1718}
{'token': '.', 'probability': 0.1121}
{'token': '=', 'probability': 0.0423}
{'token': '+=', 'probability': 0.0316}

Enter partial Python code (or type 'exit'): int(
Top-5 Suggestions:
{'token': '<IDENTIFIER>', 'probability': 0.5227}
{'token': '(', 'probability': 0.0958}
{'token': ')', 'probability': 0.0684}
{'token': '<NUMBER>', 'probability': 0.0375}
{'token': '<UNK>', 'probability': 0.0262}

Enter partial Python code (or type 'exit'): exit
